In [2]:
import sys
from pathlib import Path

import pandas as pd

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data import iter_transaction_chunks, select_period

csv_path = project_root / "data/raw/archive/HI-Small_Trans.csv"

windows = {
    "train": (
        pd.Timestamp("2022-09-03"),
        pd.Timestamp("2022-09-07"),
    ),
    "validation": (
        pd.Timestamp("2022-09-07"),
        pd.Timestamp("2022-09-09"),
    ),
}

In [3]:
summaries = []

chunks = iter_transaction_chunks(
    csv_path,
    usecols=["timestamp", "payment_format", "is_laundering"],
)

for chunk in chunks:
    chunk["timestamp"] = pd.to_datetime(
        chunk["timestamp"],
        format="%Y/%m/%d %H:%M",
        errors="raise",
    )

    for split_name, (start, end) in windows.items():
        selected = select_period(chunk, start=start, end=end)

        if selected.empty:
            continue

        if not selected["is_laundering"].isin([0, 1]).all():
            raise ValueError("Expected laundering labels to be 0 or 1.")

        summary = (
            selected.groupby("payment_format", dropna=False)
            .agg(
                transactions=("is_laundering", "size"),
                laundering_transactions=("is_laundering", "sum"),
            )
            .reset_index()
        )
        summary["split"] = split_name
        summaries.append(summary)

if not summaries:
    raise ValueError("No transactions were found in the selected windows.")

label_counts = (
    pd.concat(summaries, ignore_index=True)
    .groupby(["split", "payment_format"], dropna=False)[
        ["transactions", "laundering_transactions"]
    ]
    .sum()
    .sort_index()
)

label_counts["laundering_rate_pct"] = (
    100
    * label_counts["laundering_transactions"]
    / label_counts["transactions"]
)

display(label_counts.round({"laundering_rate_pct": 3}))

transactions  laundering_transactions  \
split      payment_format                                          
train      ACH                   175504                     1583   
           Bitcoin                40508                       24   
           Cash                  143893                       29   
           Cheque                555975                       97   
           Credit Card           412488                       67   
           Wire                   51183                        0   
validation ACH                   116910                      890   
           Bitcoin                28643                       12   
           Cash                  103357                       25   
           Cheque                397108                       59   
           Credit Card           282817                       50   
           Wire                   36689                        0   

                           laundering_rate_pct  
split      payment_format                       
train      ACH                           0.902  
           Bitcoin                       0.059  
           Cash                          0.020  
           Cheque                        0.017  
           Credit Card                   0.016  
           Wire                          0.000  
validation ACH                           0.761  
           Bitcoin                       0.042  
           Cash                          0.024  
           Cheque                        0.015  
           Credit Card                   0.018  
           Wire                          0.000

In [4]:
label_counts.to_csv(
    project_root / "reports/lab_02_label_counts.csv"
)

### Interpretation of the label distribution

Both training and validation contain both classes. ACH accounts for
most laundering examples in both periods.

Wire has no positive examples in either period, so laundering recall
cannot be evaluated for wire.

Bitcoin has only 12 validation positives. One additional detection
changes its recall by 8.3 percentage points, so its recall estimate
should be interpreted cautiously.